# Google Maps: would this platform be flagged?

The 100 Google Maps restaurant summaries are treated as one platform and put to
the market monitor: the platform is flagged when the mean G-Eval score of its
summaries is at or below the threshold $D$ fitted in `market_monitoring.ipynb`
(`results/tables/thresholds_qwen36.json`).

The verdict is reported under each of the three review sets G-Eval scored the
summaries against (`all`, `timestamp`, `time_label`; see `README.md`), with a
bootstrap over the places showing how firmly it holds.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

N_BOOTSTRAP = 2_000
RANDOM_STATE = 42

dataset = json.loads(Path("google_maps_dataset.json").read_text(encoding="utf-8"))

# The deployment thresholds D that market_monitoring.ipynb fits for this judge.
THRESHOLDS = json.loads(
    Path(f"../results/tables/thresholds_{dataset['judge']['judge_name']}.json").read_text(encoding="utf-8"))
print(f"judge: {dataset['judge']['judge_name']} ({dataset['judge']['judge_model']})")

# One row per place, metric and review set. Places are reported by an anonymized
# label (their position in the file, as in the review IDs), never by name.
scores = pd.DataFrame([
    {"place_id": place["place_id"], "label": f"R{index:03d}",
     "metric": f"geval_{metric}", "criterion": criterion, "score": score,
     "review_count": place["n_reviews"][criterion]}
    for index, place in enumerate(dataset["places"], start=1)
    for metric, by_criterion in place["geval"].items()
    for criterion, score in by_criterion.items()
]).sort_values(["place_id", "metric", "criterion"], ignore_index=True)

judge: qwen36 (Qwen/Qwen3.6-27B)


## Verdict under each review set

In [2]:
def platform_verdict(metric, criterion):
    """The platform mean under one review set, and how often resampling the places flags it."""
    values = scores.loc[scores["metric"].eq(metric) & scores["criterion"].eq(criterion),
                        "score"].to_numpy()
    threshold = THRESHOLDS[metric]
    rng = np.random.default_rng(RANDOM_STATE)
    draws = values[rng.integers(0, len(values), (N_BOOTSTRAP, len(values)))].mean(axis=1)
    return {
        "criterion": criterion,
        "metric": metric,
        "places": len(values),
        "platform_mean": round(float(values.mean()), 4),
        "ci_low": round(float(np.percentile(draws, 2.5)), 4),
        "ci_high": round(float(np.percentile(draws, 97.5)), 4),
        "threshold": threshold,
        "margin": round(float(values.mean() - threshold), 4),
        "flagged": bool(values.mean() <= threshold),
        "flagged_in_resamples": f"{float((draws <= threshold).mean()):.0%}",
    }


verdicts = pd.DataFrame([platform_verdict(metric, criterion)
                         for metric in sorted(THRESHOLDS)
                         for criterion in sorted(scores["criterion"].unique())])
print(verdicts.to_string(index=False))
print("\nmargin = platform mean - threshold; zero or negative means flagged. flagged_in_resamples is\n"
      "the share of bootstrap resamples of the places in which the platform is flagged.")

 criterion            metric  places  platform_mean  ci_low  ci_high  threshold  margin  flagged flagged_in_resamples
       all geval_consistency     100         4.7512  4.6235   4.8520   4.644983  0.1062    False                   5%
time_label geval_consistency     100         4.7357  4.6087   4.8375   4.644983  0.0907    False                   8%
 timestamp geval_consistency     100         4.1784  3.9513   4.3935   4.644983 -0.4666     True                 100%
       all   geval_relevance     100         4.7072  4.6306   4.7744   4.805009 -0.0978     True                 100%
time_label   geval_relevance     100         4.7056  4.6302   4.7725   4.805009 -0.0994     True                 100%
 timestamp   geval_relevance     100         4.2151  4.0257   4.3976   4.805009 -0.5899     True                 100%

margin = platform mean - threshold; zero or negative means flagged. flagged_in_resamples is
the share of bootstrap resamples of the places in which the platform is flagged.


## Places at or below the threshold

G-Eval Consistency under the `time_label` review set, which the paper uses.

In [3]:
consistency = scores[scores["metric"].eq("geval_consistency") & scores["criterion"].eq("time_label")]
print(f"mean {consistency['score'].mean():.4f}   median {consistency['score'].median():.4f}   "
      f"at or below threshold: {int((consistency['score'] <= THRESHOLDS['geval_consistency']).sum())}"
      f"/{len(consistency)} places")

mean 4.7357   median 4.9627   at or below threshold: 16/100 places


## Summary lengths

Google's summaries against the summarization styles of the experiments, in words. Short summaries omit more of the reviews' content, which lowers G-Eval Relevance (Fig. 4: held-out single-sentence summaries).

In [4]:
summaries = pd.read_json(Path("../data/summaries.jsonl"), lines=True)
summaries = summaries[summaries["record_id"].eq(summaries["parent_summary_id"])]
lengths = summaries.groupby("summary_condition")["text"].apply(lambda t: t.str.split().str.len().mean())
google_lengths = pd.Series([len(place["summary"].split()) for place in dataset["places"]])

print(f"Google Maps summaries: {google_lengths.mean():.1f} words on average "
      f"(median {google_lengths.median():.0f}, range {google_lengths.min()}-{google_lengths.max()})\n")
print("Experiment styles, mean words:")
print(lengths.sort_values().round(1).to_string())

Google Maps summaries: 40.2 words on average (median 40, range 21-60)

Experiment styles, mean words:
summary_condition
openai_single_sentence           31.1
gemini_faithful                  86.0
openai_gmaps_style               86.6
openai_majority_opinion          91.1
openai_short_summary             92.8
openai_positive_and_negative     95.0
openai_amazon_style              96.2
qwen_faithful                   100.0
claude_faithful                 102.5
openai_bullet_points            119.5
openai_long_summary             222.6
